VI.b) Classify the 251 batch-adjusted Human-Chimpanzee DE genes by gene biotype

This notebook reads the batch-adjusted differential-expression results, extracts the selected 251 genes, annotates them with current human gene-type information using MyGene.info, groups them into broad classes, and checks the genes specifically mentioned in the review.


In [1]:
from pathlib import Path
import sys
import subprocess
import pandas as pd

try:
    import mygene
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'mygene'])
    import mygene

RESULTS_FILE = Path('results/early_human_chimp_pylimma_results_human_spec_batch_adjusted.csv')
OUT_FULL = Path('results/early_human_chimp_251_gene_biotypes_python.csv')
OUT_SUMMARY = Path('results/early_human_chimp_251_gene_biotype_summary_python.csv')

print('Setup complete.')


Setup complete.


Load the batch-adjusted DE results and extract the selected genes.


In [2]:
if not RESULTS_FILE.exists():
    raise FileNotFoundError(f'Missing file: {RESULTS_FILE}')

de_results = pd.read_csv(RESULTS_FILE)

required = {'Gene', 'selected'}
missing = required - set(de_results.columns)
if missing:
    raise ValueError('Missing required columns: ' + ', '.join(sorted(missing)))

selected_flag = (
    de_results['selected']
    .astype(str)
    .str.strip()
    .str.upper()
    .isin({'TRUE', '1'})
)

selected = de_results.loc[selected_flag].copy()
selected['Gene'] = selected['Gene'].astype(str).str.strip().str.upper()
selected = selected.drop_duplicates(subset='Gene').reset_index(drop=True)

genes251 = selected['Gene'].tolist()

print('Selected genes:', len(genes251))
selected.head()


Selected genes: 251


,Gene,mean_expression,human_minus_chimp_log2_difference,posterior_residual_sd,standardized_effect,moderated_t_zero,p_value_zero,fdr_zero,treat_t,treat_p_value,treat_fdr,loo_direction_matches,loo_direction_fraction,direction,selected,rank
0,TMSB4XP2,4.565120,4.632113,0.335915,13.789520,18.054700,4.621908e-09,0.000054,16.885383,5.717076e-09,0.000067,9,1.0,Higher in Human,True,1
1,BNIP3P1,2.869818,-5.054740,0.439246,-11.507757,-15.067174,2.739106e-08,0.000160,-14.172934,3.265499e-08,0.000190,9,1.0,Higher in Chimpanzee,True,2
2,PDIA2,2.240149,3.789176,0.377124,10.047568,13.155340,1.023989e-07,0.000352,12.113794,1.376163e-07,0.000535,9,1.0,Higher in Human,True,3
3,LINC00909,2.879049,2.453054,0.248321,9.878575,12.934077,1.206064e-07,0.000352,11.352284,2.297998e-07,0.000670,9,1.0,Higher in Human,True,4
4,COX7A2P2,4.275137,-3.194790,0.362676,-8.808938,-11.533594,3.614604e-07,0.000802,-10.450556,5.353742e-07,0.001249,9,1.0,Higher in Chimpanzee,True,5


Query current human gene-type annotations using MyGene.info.


In [3]:
mg = mygene.MyGeneInfo()

# Annotate the full tested universe so that we can export a protein-coding
# background list for GOfuncR as well as the protein-coding selected genes.
all_genes = (
    de_results['Gene']
    .astype(str)
    .str.strip()
    .str.upper()
    .drop_duplicates()
    .tolist()
)

query = mg.querymany(
    all_genes,
    scopes='symbol',
    fields='symbol,type_of_gene,ensembl.gene',
    species='human',
    as_dataframe=False,
    returnall=False,
    verbose=False,
)

records = []
for item in query:
    records.append({
        'query': item.get('query'),
        'matched_symbol': item.get('symbol'),
        'gene_biotype': item.get('type_of_gene'),
        'notfound': item.get('notfound', False),
    })

ann = pd.DataFrame(records).drop_duplicates(subset='query', keep='first')

# Annotation for all tested genes.
all_classified = pd.DataFrame({'Gene': all_genes}).merge(
    ann,
    how='left',
    left_on='Gene',
    right_on='query',
    sort=False,
)

# Annotation for the 251 selected genes.
classified = selected.merge(
    ann,
    how='left',
    left_on='Gene',
    right_on='query',
    sort=False,
)

classified.head()


,Gene,mean_expression,human_minus_chimp_log2_difference,posterior_residual_sd,standardized_effect,moderated_t_zero,p_value_zero,fdr_zero,treat_t,treat_p_value,treat_fdr,loo_direction_matches,loo_direction_fraction,direction,selected,rank,query,matched_symbol,gene_biotype,notfound
0,TMSB4XP2,4.565120,4.632113,0.335915,13.789520,18.054700,4.621908e-09,0.000054,16.885383,5.717076e-09,0.000067,9,1.0,Higher in Human,True,1,TMSB4XP2,TMSB4XP2,NaN,False
1,BNIP3P1,2.869818,-5.054740,0.439246,-11.507757,-15.067174,2.739106e-08,0.000160,-14.172934,3.265499e-08,0.000190,9,1.0,Higher in Chimpanzee,True,2,BNIP3P1,BNIP3P1,pseudo,False
2,PDIA2,2.240149,3.789176,0.377124,10.047568,13.155340,1.023989e-07,0.000352,12.113794,1.376163e-07,0.000535,9,1.0,Higher in Human,True,3,PDIA2,PDIA2,protein-coding,False
3,LINC00909,2.879049,2.453054,0.248321,9.878575,12.934077,1.206064e-07,0.000352,11.352284,2.297998e-07,0.000670,9,1.0,Higher in Human,True,4,LINC00909,ZNF407-AS1,ncRNA,False
4,COX7A2P2,4.275137,-3.194790,0.362676,-8.808938,-11.533594,3.614604e-07,0.000802,-10.450556,5.353742e-07,0.001249,9,1.0,Higher in Chimpanzee,True,5,COX7A2P2,COX7A2P2,NaN,False


Collapse detailed gene types into broad categories relevant to the review.


In [4]:
def collapse_biotype(bt):
    if pd.isna(bt) or str(bt).strip() == '':
        return 'Unmapped'

    x = str(bt).lower()

    if 'pseudo' in x:
        return 'Pseudogene'

    if any(term in x for term in (
        'lnc', 'non-coding', 'noncoding', 'antisense',
        'ncrna', 'mirna', 'snrna', 'snorna', 'rrna', 'misc_rna'
    )):
        return 'lncRNA/non-coding'

    if x in {'protein-coding', 'protein_coding'}:
        return 'Protein-coding'

    return 'Other'

classified['class'] = classified['gene_biotype'].apply(collapse_biotype)
all_classified['class'] = all_classified['gene_biotype'].apply(collapse_biotype)

summary = (
    classified['class']
    .value_counts(dropna=False)
    .rename_axis('class')
    .reset_index(name='n_genes')
)

summary['percent'] = (100 * summary['n_genes'] / len(classified)).round(1)
summary


,class,n_genes,percent
0,Protein-coding,225,89.6
1,Pseudogene,10,4.0
2,lncRNA/non-coding,9,3.6
3,Unmapped,7,2.8


Inspect the genes specifically mentioned in the review.


In [5]:
review_genes = [
    'TMSB4XP2',
    'BNIP3P1',
    'COX7A2P2',
    'RPL7P17',
    'TPMTP1',
]

review_cols = [c for c in [
    'Gene',
    'matched_symbol',
    'gene_biotype',
    'class',
    'direction',
    'human_minus_chimp_log2_difference',
    'treat_fdr',
] if c in classified.columns]

classified.loc[classified['Gene'].isin(review_genes), review_cols]


,Gene,matched_symbol,gene_biotype,class,direction,human_minus_chimp_log2_difference,treat_fdr
0,TMSB4XP2,TMSB4XP2,NaN,Unmapped,Higher in Human,4.632113,0.000067
1,BNIP3P1,BNIP3P1,pseudo,Pseudogene,Higher in Chimpanzee,-5.054740,0.000190
4,COX7A2P2,COX7A2P2,NaN,Unmapped,Higher in Chimpanzee,-3.194790,0.001249
5,RPL7P17,RPL7P17,pseudo,Pseudogene,Higher in Human,3.195221,0.001364
16,TPMTP1,TPMTP1,NaN,Unmapped,Higher in Chimpanzee,-2.183842,0.007162


Export protein-coding gene lists for the GOfuncR sensitivity analysis.

The first list contains all protein-coding genes in the full tested gene universe. The second list contains the protein-coding genes among the 251 batch-adjusted selected genes. These lists can be read directly into the GOfuncR notebook.


In [6]:
FULL_PC_FILE = Path('results/protein_coding_genes_full_tested_universe.csv')
SELECTED_PC_FILE = Path('results/protein_coding_genes_selected_251.csv')

protein_coding_all = (
    all_classified.loc[all_classified['class'] == 'Protein-coding', ['Gene']]
    .drop_duplicates()
    .sort_values('Gene')
    .reset_index(drop=True)
)

protein_coding_selected = (
    classified.loc[classified['class'] == 'Protein-coding', ['Gene']]
    .drop_duplicates()
    .sort_values('Gene')
    .reset_index(drop=True)
)

protein_coding_all.to_csv(FULL_PC_FILE, index=False)
protein_coding_selected.to_csv(SELECTED_PC_FILE, index=False)

print('Protein-coding genes in full tested universe:', len(protein_coding_all))
print('Protein-coding genes among 251 selected genes:', len(protein_coding_selected))
print('\nSaved:')
print(FULL_PC_FILE)
print(SELECTED_PC_FILE)

protein_coding_selected.head(20)


Protein-coding genes in full tested universe: 11052
Protein-coding genes among 251 selected genes: 225

Saved:
results/protein_coding_genes_full_tested_universe.csv
results/protein_coding_genes_selected_251.csv


,Gene
0,ABTB1
1,ADAM20
2,ADCYAP1
3,AGPAT5
4,ALG3
5,ANKRD29
6,AP1S3
7,ASAP1
8,ATP10B
9,ATP5D


Save the detailed annotation table and the summary.


In [7]:
OUT_FULL.parent.mkdir(parents=True, exist_ok=True)

classified.to_csv(OUT_FULL, index=False)
summary.to_csv(OUT_SUMMARY, index=False)
all_classified.to_csv(
    'results/full_tested_gene_universe_biotypes_python.csv',
    index=False,
)

print('Saved:')
print(OUT_FULL)
print(OUT_SUMMARY)
print('results/full_tested_gene_universe_biotypes_python.csv')
print(FULL_PC_FILE)
print(SELECTED_PC_FILE)


Saved:
results/early_human_chimp_251_gene_biotypes_python.csv
results/early_human_chimp_251_gene_biotype_summary_python.csv
results/full_tested_gene_universe_biotypes_python.csv
results/protein_coding_genes_full_tested_universe.csv
results/protein_coding_genes_selected_251.csv
